# 03. 알고리즘 비교 — PPO vs A2C vs DQN

강화학습에는 여러 알고리즘이 있고, 문제에 따라 성능과 학습 효율이 다릅니다.
이 노트북에서는 CartPole에서 대표 알고리즘 3가지를 같은 조건으로 학습해 비교합니다.

- **PPO** (Proximal Policy Optimization) — 정책 기반, 안정적
- **A2C** (Advantage Actor-Critic) — 정책 기반, 가벼움
- **DQN** (Deep Q-Network) — 가치 기반

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.evaluation import evaluate_policy

plt.rc('font', family='NanumGothic')
plt.rcParams['axes.unicode_minus'] = False

## 1. 동일 조건으로 세 알고리즘 학습

같은 환경, 같은 학습 스텝으로 세 알고리즘을 각각 학습하고 평균 보상을 측정합니다.
(학습 스텝을 늘리면 결과가 더 안정적입니다.)

In [ ]:
algorithms = {'PPO': PPO, 'A2C': A2C, 'DQN': DQN}
TIMESTEPS = 30000
results = {}

for name, Algo in algorithms.items():
    env = gym.make('CartPole-v1')
    model = Algo('MlpPolicy', env, verbose=0)
    model.learn(total_timesteps=TIMESTEPS)
    mean_r, std_r = evaluate_policy(model, env, n_eval_episodes=20)
    results[name] = (mean_r, std_r)
    print(f'{name}: 평균 보상 {mean_r:.1f} +/- {std_r:.1f}')
    env.close()

## 2. 결과 비교

In [ ]:
names = list(results.keys())
means = [results[n][0] for n in names]
stds = [results[n][1] for n in names]

plt.figure(figsize=(6, 4))
plt.bar(names, means, yerr=stds, capsize=5, color=['#4c72b0', '#dd8452', '#55a868'])
plt.ylabel('평균 보상'); plt.title('알고리즘별 성능 (CartPole, 동일 학습 스텝)')
plt.ylim(0, 520)
for i, v in enumerate(means):
    plt.text(i, v + 10, f'{v:.0f}', ha='center')
plt.tight_layout(); plt.show()

### 정리

- 같은 조건에서 PPO·A2C·DQN을 학습해 성능을 비교했습니다.
- 동일한 학습 스텝에서도 알고리즘마다 수렴 속도와 안정성이 다릅니다 — 문제 특성에 맞는 알고리즘 선택이 중요합니다.
- 학습 스텝, 하이퍼파라미터, 환경을 바꿔가며 실험해 보세요. (TensorBoard 로그는 `/workspace/runs` 등에 저장해 비교할 수 있습니다.)